In [0]:
%run ../../src/utils/config

In [0]:
%python
import os

config_path = os.path.abspath(
    os.path.join(os.getcwd(), "../../config/config.yml")
)

config = load_config(config_path)

catalog = config["target"]["catalog"]
silver_schema = config["target"]["silver_schema"]
gold_schema = config["target"]["gold_schema"]

print(f"Catalog: {catalog}")
print(f"Silver schema: {silver_schema}")
print(f"Gold schema: {gold_schema}")

In [0]:
%python
from pyspark.sql import functions as F

silver_stats = spark.table(
    f"{catalog}.{silver_schema}.pokemon_stats"
)

dim_pokemon = (
    spark.table(f"{catalog}.{gold_schema}.dim_pokemon")
    .filter(F.col("is_current") == True)
    .select("pokemon_id", "pokemon_key")
)

dim_stat = (
    spark.table(f"{catalog}.{gold_schema}.dim_stat")
    .select("stat_url", "stat_key")
)

print(f"Silver stats rows: {silver_stats.count()}")
print(f"Pokémon dimension rows: {dim_pokemon.count()}")
print(f"Stat dimension rows: {dim_stat.count()}")

In [0]:
%python
stats_source = (
    silver_stats
    .join(dim_pokemon, on="pokemon_id", how="left")
    .join(dim_stat, on="stat_url", how="left")
    .select(
        "pokemon_key",
        "stat_key",
        "base_stat",
        "effort"
    )
)

print(f"Prepared fact rows: {stats_source.count()}")

display(stats_source.limit(10))

In [0]:
%python
missing_keys = stats_source.filter(
    F.col("pokemon_key").isNull() |
    F.col("stat_key").isNull()
).count()

if missing_keys > 0:
    raise ValueError(
        f"fact_pokemon_stats: {missing_keys} unresolved dimension keys."
    )

duplicate_count = (
    stats_source
    .groupBy("pokemon_key", "stat_key")
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
)

if duplicate_count > 0:
    raise ValueError(
        "fact_pokemon_stats: duplicate fact grain detected."
    )

print("fact_pokemon_stats source validation PASSED")

In [0]:
%python
target_table = f"{catalog}.{gold_schema}.fact_pokemon_stats"

stats_source.createOrReplaceTempView("src_fact_pokemon_stats")

spark.sql(f"""
    MERGE INTO {target_table} AS tgt
    USING src_fact_pokemon_stats AS src
       ON tgt.pokemon_key = src.pokemon_key
      AND tgt.stat_key = src.stat_key

    WHEN MATCHED AND (
        NOT (tgt.base_stat <=> src.base_stat)
        OR NOT (tgt.effort <=> src.effort)
    )
    THEN UPDATE SET
        tgt.base_stat = src.base_stat,
        tgt.effort = src.effort,
        tgt.gold_loaded_at = current_timestamp()

    WHEN NOT MATCHED THEN INSERT (
        pokemon_key,
        stat_key,
        base_stat,
        effort,
        gold_loaded_at
    )
    VALUES (
        src.pokemon_key,
        src.stat_key,
        src.base_stat,
        src.effort,
        current_timestamp()
    )
""")

print("fact_pokemon_stats load completed.")

In [0]:
%python
loaded = spark.table(
    f"{catalog}.{gold_schema}.fact_pokemon_stats"
)

null_key_count = loaded.filter(
    F.col("pokemon_key").isNull() |
    F.col("stat_key").isNull()
).count()

duplicate_count = (
    loaded
    .groupBy("pokemon_key", "stat_key")
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
)

null_timestamp_count = loaded.filter(
    F.col("gold_loaded_at").isNull()
).count()

if null_key_count > 0:
    raise ValueError("Null dimension keys found.")

if duplicate_count > 0:
    raise ValueError("Duplicate fact grain found.")

if null_timestamp_count > 0:
    raise ValueError("Null gold_loaded_at found.")

print(f"Fact rows: {loaded.count()}")
print("fact_pokemon_stats validation PASSED")

In [0]:
%python
silver_move_details = spark.table(
    f"{catalog}.{silver_schema}.pokemon_move_details"
)

dim_pokemon = (
    spark.table(f"{catalog}.{gold_schema}.dim_pokemon")
    .filter(F.col("is_current") == True)
    .select("pokemon_id", "pokemon_key")
)

dim_move = spark.table(
    f"{catalog}.{gold_schema}.dim_move"
).select("move_url", "move_key")

dim_version_group = spark.table(
    f"{catalog}.{gold_schema}.dim_version_group"
).select("version_group_url", "version_group_key")

dim_learn_method = spark.table(
    f"{catalog}.{gold_schema}.dim_learn_method"
).select("learn_method_url", "learn_method_key")

print(f"Silver move detail rows: {silver_move_details.count()}")

In [0]:
%python
from pyspark.sql import functions as F

s = silver_move_details.alias("s")

p = (
    spark.table(f"{catalog}.{gold_schema}.dim_pokemon")
    .filter(F.col("is_current") == True)
    .select("pokemon_id", "pokemon_key")
    .alias("p")
)

m = (
    spark.table(f"{catalog}.{gold_schema}.dim_move")
    .select("move_name", "move_key")
    .alias("m")
)

vg = dim_version_group.alias("vg")
lm = dim_learn_method.alias("lm")

move_learning_source = (
    s
    .join(p, F.col("s.pokemon_id") == F.col("p.pokemon_id"), "left")
    .join(m, F.col("s.move_name") == F.col("m.move_name"), "left")
    .join(vg, F.col("s.version_group_url") == F.col("vg.version_group_url"), "left")
    .join(lm, F.col("s.learn_method_url") == F.col("lm.learn_method_url"), "left")
    .select(
        F.col("p.pokemon_key").alias("pokemon_key"),
        F.col("m.move_key").alias("move_key"),
        F.col("vg.version_group_key").alias("version_group_key"),
        F.col("lm.learn_method_key").alias("learn_method_key"),
        F.col("s.level_learned_at").alias("level_learned_at"),
        F.col("s.learn_order").alias("learn_order")
    )
)

print(f"Silver source rows: {silver_move_details.count()}")
print(f"Prepared fact rows: {move_learning_source.count()}")

display(move_learning_source.limit(10))

In [0]:
%python
from pyspark.sql import functions as F

s = silver_move_details.alias("s")

p = (
    spark.table(f"{catalog}.{gold_schema}.dim_pokemon")
    .filter(F.col("is_current") == True)
    .select("pokemon_id", "pokemon_key")
    .alias("p")
)

m = (
    spark.table(f"{catalog}.{gold_schema}.dim_move")
    .select("move_name", "move_key")
    .alias("m")
)

vg = dim_version_group.alias("vg")
lm = dim_learn_method.alias("lm")

move_learning_source = (
    s
    .join(p, F.col("s.pokemon_id") == F.col("p.pokemon_id"), "left")
    .join(m, F.col("s.move_name") == F.col("m.move_name"), "left")
    .join(vg, F.col("s.version_group_url") == F.col("vg.version_group_url"), "left")
    .join(lm, F.col("s.learn_method_url") == F.col("lm.learn_method_url"), "left")
    .select(
        F.col("p.pokemon_key").alias("pokemon_key"),
        F.col("m.move_key").alias("move_key"),
        F.col("vg.version_group_key").alias("version_group_key"),
        F.col("lm.learn_method_key").alias("learn_method_key"),
        F.col("s.level_learned_at").alias("level_learned_at"),
        F.col("s.learn_order").alias("learn_order")
    )
)

print(f"Silver source rows: {silver_move_details.count()}")
print(f"Prepared fact rows: {move_learning_source.count()}")

display(move_learning_source.limit(10))

In [0]:
%python
from pyspark.sql import functions as F

fact_grain = [
    "pokemon_key",
    "move_key",
    "version_group_key",
    "learn_method_key",
    "level_learned_at",
    "learn_order"
]

# Check missing dimension keys
key_columns = [
    "pokemon_key",
    "move_key",
    "version_group_key",
    "learn_method_key"
]

for column in key_columns:
    missing = move_learning_source.filter(
        F.col(column).isNull()
    ).count()

    print(f"{column} missing: {missing}")

    if missing > 0:
        raise ValueError(f"Missing keys in {column}")

# Check duplicate fact grain
duplicate_grains = (
    move_learning_source
    .groupBy(*fact_grain)
    .count()
    .filter(F.col("count") > 1)
)

duplicate_count = duplicate_grains.count()

print(f"Duplicate fact-grain combinations: {duplicate_count}")

if duplicate_count > 0:
    duplicate_grains.orderBy(
        F.col("count").desc()
    ).show(20, truncate=False)

    raise ValueError("Duplicate fact grain detected")

print(f"Validated fact rows: {move_learning_source.count()}")
print("Fact source validation PASSED")

In [0]:
%python
from delta.tables import DeltaTable
from pyspark.sql import functions as F

target_table = f"{catalog}.{gold_schema}.fact_pokemon_move_learning"

target = DeltaTable.forName(spark, target_table)

(
    target.alias("t")
    .merge(
        move_learning_source.alias("s"),
        """
        t.pokemon_key = s.pokemon_key
        AND t.move_key = s.move_key
        AND t.version_group_key = s.version_group_key
        AND t.learn_method_key = s.learn_method_key
        AND t.level_learned_at <=> s.level_learned_at
        AND t.learn_order <=> s.learn_order
        """
    )
    .whenMatchedUpdate(
        set={
            "level_learned_at": "s.level_learned_at",
            "learn_order": "s.learn_order",
            "gold_loaded_at": "current_timestamp()"
        }
    )
    .whenNotMatchedInsert(
        values={
            "pokemon_key": "s.pokemon_key",
            "move_key": "s.move_key",
            "version_group_key": "s.version_group_key",
            "learn_method_key": "s.learn_method_key",
            "level_learned_at": "s.level_learned_at",
            "learn_order": "s.learn_order",
            "gold_loaded_at": "current_timestamp()"
        }
    )
    .execute()
)

print("fact_pokemon_move_learning loaded successfully.")

In [0]:
%python
from pyspark.sql import functions as F

fact_table = spark.table(
    f"{catalog}.{gold_schema}.fact_pokemon_move_learning"
)

fact_grain = [
    "pokemon_key",
    "move_key",
    "version_group_key",
    "learn_method_key",
    "level_learned_at",
    "learn_order"
]

row_count = fact_table.count()

duplicate_count = (
    fact_table
    .groupBy(*fact_grain)
    .count()
    .filter(F.col("count") > 1)
    .count()
)

null_key_count = fact_table.filter(
    F.col("pokemon_key").isNull()
    | F.col("move_key").isNull()
    | F.col("version_group_key").isNull()
    | F.col("learn_method_key").isNull()
).count()

null_timestamp_count = fact_table.filter(
    F.col("gold_loaded_at").isNull()
).count()

print(f"Fact rows: {row_count}")
print(f"Duplicate fact grains: {duplicate_count}")
print(f"Rows with null dimension keys: {null_key_count}")
print(f"Rows with null gold_loaded_at: {null_timestamp_count}")

if any([
    row_count != 638321,
    duplicate_count > 0,
    null_key_count > 0,
    null_timestamp_count > 0
]):
    raise ValueError("fact_pokemon_move_learning validation FAILED")

print("fact_pokemon_move_learning validation PASSED")

In [0]:
%python
silver_game_indices = spark.table(
    f"{catalog}.{silver_schema}.pokemon_game_indices"
)

dim_pokemon = (
    spark.table(f"{catalog}.{gold_schema}.dim_pokemon")
    .filter(F.col("is_current") == True)
    .select("pokemon_id", "pokemon_key")
)

dim_game_version = (
    spark.table(f"{catalog}.{gold_schema}.dim_game_version")
    .select("version_url", "game_version_key")
)

print("Silver rows:", silver_game_indices.count())
print("Game version dimension rows:", dim_game_version.count())

In [0]:
%python
game_presence_source = (
    silver_game_indices.alias("s")
    .join(
        dim_pokemon.alias("p"),
        F.col("s.pokemon_id") == F.col("p.pokemon_id"),
        "left"
    )
    .join(
        dim_game_version.alias("g"),
        F.col("s.version_url") == F.col("g.version_url"),
        "left"
    )
    .select(
        F.col("p.pokemon_key").alias("pokemon_key"),
        F.col("g.game_version_key").alias("game_version_key"),
        F.col("s.game_index").alias("game_index")
    )
)

print("Prepared fact rows:", game_presence_source.count())
display(game_presence_source.limit(10))

In [0]:
%python
fact_grain = ["pokemon_key", "game_version_key"]

for column in fact_grain:
    missing = game_presence_source.filter(
        F.col(column).isNull()
    ).count()

    print(f"{column} missing: {missing}")

    if missing > 0:
        raise ValueError(f"Missing dimension key: {column}")

duplicate_count = (
    game_presence_source
    .groupBy(*fact_grain)
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Duplicate fact-grain combinations:", duplicate_count)

if duplicate_count > 0:
    raise ValueError("Duplicate fact grain detected")

print("fact_pokemon_game_presence source validation PASSED")

In [0]:
%python
from delta.tables import DeltaTable

target_table = f"{catalog}.{gold_schema}.fact_pokemon_game_presence"

target = DeltaTable.forName(spark, target_table)

(
    target.alias("t")
    .merge(
        game_presence_source.alias("s"),
        """
        t.pokemon_key = s.pokemon_key
        AND t.game_version_key = s.game_version_key
        """
    )
    .whenMatchedUpdate(
        condition="NOT (t.game_index <=> s.game_index)",
        set={
            "game_index": "s.game_index",
            "gold_loaded_at": "current_timestamp()"
        }
    )
    .whenNotMatchedInsert(
        values={
            "pokemon_key": "s.pokemon_key",
            "game_version_key": "s.game_version_key",
            "game_index": "s.game_index",
            "gold_loaded_at": "current_timestamp()"
        }
    )
    .execute()
)

print("fact_pokemon_game_presence loaded successfully.")

In [0]:
%python
from pyspark.sql import functions as F

fact_table = spark.table(
    f"{catalog}.{gold_schema}.fact_pokemon_game_presence"
)

fact_grain = ["pokemon_key", "game_version_key"]

row_count = fact_table.count()

duplicate_count = (
    fact_table
    .groupBy(*fact_grain)
    .count()
    .filter(F.col("count") > 1)
    .count()
)

null_key_count = fact_table.filter(
    F.col("pokemon_key").isNull()
    | F.col("game_version_key").isNull()
).count()

null_timestamp_count = fact_table.filter(
    F.col("gold_loaded_at").isNull()
).count()

print(f"Fact rows: {row_count}")
print(f"Duplicate fact grains: {duplicate_count}")
print(f"Rows with null dimension keys: {null_key_count}")
print(f"Rows with null gold_loaded_at: {null_timestamp_count}")

if any([
    duplicate_count > 0,
    null_key_count > 0,
    null_timestamp_count > 0
]):
    raise ValueError("fact_pokemon_game_presence validation FAILED")

print("fact_pokemon_game_presence validation PASSED")

In [0]:
%python
from pyspark.sql import functions as F

gold_tables = {
    "dim_pokemon": "pokemon_key",
    "dim_type": "type_key",
    "dim_ability": "ability_key",
    "dim_move": "move_key",
    "dim_form": "form_key",
    "dim_version_group": "version_group_key",
    "dim_learn_method": "learn_method_key",
    "dim_game_version": "game_version_key",
    "dim_stat": "stat_key"
}

# Load dimension keys
dimension_keys = {
    name: spark.table(f"{catalog}.{gold_schema}.{name}")
    .select(key).distinct()
    for name, key in gold_tables.items()
}

# Define foreign-key relationships
relationships = {
    "bridge_pokemon_type": {
        "pokemon_key": "dim_pokemon",
        "type_key": "dim_type"
    },
    "bridge_pokemon_ability": {
        "pokemon_key": "dim_pokemon",
        "ability_key": "dim_ability"
    },
    "bridge_pokemon_form": {
        "pokemon_key": "dim_pokemon",
        "form_key": "dim_form"
    },
    "fact_pokemon_stats": {
        "pokemon_key": "dim_pokemon",
        "stat_key": "dim_stat"
    },
    "fact_pokemon_move_learning": {
        "pokemon_key": "dim_pokemon",
        "move_key": "dim_move",
        "version_group_key": "dim_version_group",
        "learn_method_key": "dim_learn_method"
    },
    "fact_pokemon_game_presence": {
        "pokemon_key": "dim_pokemon",
        "game_version_key": "dim_game_version"
    }
}

for table_name, fk_map in relationships.items():

    df = spark.table(f"{catalog}.{gold_schema}.{table_name}")

    for fk_column, dimension_name in fk_map.items():

        dimension_key = gold_tables[dimension_name]

        missing = (
            df.select(fk_column)
            .distinct()
            .join(
                dimension_keys[dimension_name],
                df[fk_column] == dimension_keys[dimension_name][dimension_key],
                "left_anti"
            )
            .count()
        )

        print(f"{table_name}.{fk_column}: orphan keys = {missing}")

        if missing > 0:
            raise ValueError(
                f"Referential integrity failed: {table_name}.{fk_column}"
            )

print("Gold-layer referential integrity validation PASSED")

In [0]:
%python
from pyspark.sql import functions as F

# 1. Validate dimension keys and natural keys
dimension_checks = {
    "dim_pokemon": {
        "surrogate": "pokemon_key",
        "natural": "pokemon_id",
        "scd2": True
    },
    "dim_type": {
        "surrogate": "type_key",
        "natural": "type_url",
        "scd2": False
    },
    "dim_ability": {
        "surrogate": "ability_key",
        "natural": "ability_url",
        "scd2": False
    },
    "dim_move": {
        "surrogate": "move_key",
        "natural": "move_url",
        "scd2": False
    },
    "dim_form": {
        "surrogate": "form_key",
        "natural": "form_url",
        "scd2": False
    },
    "dim_version_group": {
        "surrogate": "version_group_key",
        "natural": "version_group_url",
        "scd2": False
    },
    "dim_learn_method": {
        "surrogate": "learn_method_key",
        "natural": "learn_method_url",
        "scd2": False
    },
    "dim_game_version": {
        "surrogate": "game_version_key",
        "natural": "version_url",
        "scd2": False
    },
    "dim_stat": {
        "surrogate": "stat_key",
        "natural": "stat_url",
        "scd2": False
    }
}

for table_name, checks in dimension_checks.items():

    df = spark.table(f"{catalog}.{gold_schema}.{table_name}")

    surrogate = checks["surrogate"]
    natural = checks["natural"]

    null_surrogate = df.filter(F.col(surrogate).isNull()).count()

    duplicate_surrogate = (
        df.groupBy(surrogate)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    if checks["scd2"]:
        natural_df = df.filter(F.col("is_current") == True)
    else:
        natural_df = df

    duplicate_natural = (
        natural_df.groupBy(natural)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    print(
        f"{table_name}: "
        f"rows={df.count()}, "
        f"null_surrogate={null_surrogate}, "
        f"duplicate_surrogate={duplicate_surrogate}, "
        f"duplicate_natural={duplicate_natural}"
    )

    if null_surrogate or duplicate_surrogate or duplicate_natural:
        raise ValueError(f"Dimension validation failed: {table_name}")


# 2. Validate Pokémon SCD Type 2
pokemon_dim = spark.table(
    f"{catalog}.{gold_schema}.dim_pokemon"
)

invalid_current = pokemon_dim.filter(
    (F.col("is_current") == True)
    & F.col("effective_to").isNotNull()
).count()

invalid_historical = pokemon_dim.filter(
    (F.col("is_current") == False)
    & F.col("effective_to").isNull()
).count()

if invalid_current or invalid_historical:
    raise ValueError("dim_pokemon SCD2 validation failed")

print("dim_pokemon SCD2 validation PASSED")


# 3. Print all Gold table counts
gold_table_names = [
    *dimension_checks.keys(),
    "bridge_pokemon_type",
    "bridge_pokemon_ability",
    "bridge_pokemon_form",
    "fact_pokemon_stats",
    "fact_pokemon_move_learning",
    "fact_pokemon_game_presence"
]

print("\n--- GOLD TABLE SUMMARY ---")

for table_name in gold_table_names:
    count = spark.table(
        f"{catalog}.{gold_schema}.{table_name}"
    ).count()

    print(f"{table_name}: {count}")

print("\nCOMPLETE GOLD-LAYER AUDIT PASSED")